# UrgeNurse — Benchmark de modelos ASR (speech-to-text) en CPU

Este notebook usa `asr.py` como **paquete de funciones** para:

- **descargar** los modelos ASR del catálogo (faster-whisper, whisper.cpp,
  Parakeet, Moonshine, Vosk) — CPU, cuantizados, ≤ 2 GB RAM,
- **correr** cada modelo sobre los 100 audios WAV (16 kHz mono 16-bit),
  escribiendo una transcripción por audio en `transcriptions/<modelo>/{n}.txt`,
- **calcular y analizar las métricas**: WER global y de terminología de
  enfermería, CER, RTF, RAM peak, NER recall, latencia/duración y alucinación.

> **Prerrequisito:** las referencias (ground-truth verbatim) deben existir ya en
> `assets/references/`. Se generan **una sola vez** con el comando aparte:
> ```bash
> python prepare_references.py
> ```
> que convierte los perfiles `.docx` → `.txt` y transcribe cada audio con Whisper
> large-v3. Si faltan, `run_benchmark()` avisa y se detiene.

> **Alucinación de Whisper** (riesgo documentado por el MIT en silencios y ruido):
> se mitiga con **filtro VAD Silero** + **umbral de confianza por palabra** +
> **detección de repeticiones/frases fantasma**, y se reporta `hallucination_rate`.
>
> **Sesgo declarado:** la ground-truth la produce Whisper large-v3, así que la
> comparación favorece ligeramente a la familia Whisper. Debe constar en la memoria.

Requiere `faster-whisper`, `pandas`, `matplotlib`. Backends opcionales
(`pywhispercpp`, `sherpa-onnx`, `vosk`) se omiten solos si no están instalados.

In [ ]:
# 1) Configuración por variables de entorno (se leen al importar asr)
import os
from pathlib import Path

MODELS_DIR = (Path.cwd() / ".." / "models").resolve()
os.environ["ASR_MODELS_DIR"] = str(MODELS_DIR)
os.environ["ASR_MAX_RAM_GB"] = (
    "2.0"  # presupuesto de RAM para los modelos del benchmark
)
os.environ["ASR_LIMIT"] = "0"  # 0 = los 100 audios; baja a 20 para iterar rápido
os.environ["ASR_VAD"] = "1"  # filtro VAD anti-alucinación (Whisper)
os.environ["ASR_DOWNLOAD"] = "1"  # descarga modelos/artefactos que falten

for k in ("ASR_MODELS_DIR", "ASR_MAX_RAM_GB", "ASR_LIMIT", "ASR_VAD"):
    print(f"{k:16s}= {os.environ[k]}")

In [ ]:
# 2) Importar asr.py (añadimos su carpeta al sys.path para un import robusto)
import sys
from pathlib import Path

ASR_DIR = next(
    p
    for p in (Path.cwd(), Path.cwd() / "scripts" / "asr", Path.cwd().parent)
    if (p / "asr.py").exists()
)
if str(ASR_DIR) not in sys.path:
    sys.path.insert(0, str(ASR_DIR))

import asr

print(f"asr.py cargado desde: {ASR_DIR}")
print(f"Modelos en el catálogo ({len(asr.MODELS)}):")
for m in asr.MODELS:
    tag = " (baseline)" if m.is_baseline else ""
    print(f"  · {m.name:<32} [{m.backend}]{tag}")

In [ ]:
# 2b) Información del computador (reusa el helper del benchmark de LLMs)
sysinfo = asr.print_system_info()

## Correr el benchmark → DataFrame

`asr.run_benchmark()` descarga y carga cada modelo (CPU, ≤ 2 GB), transcribe los
100 audios, escribe `transcriptions/<modelo>/{n}.txt` por audio y devuelve un
`DataFrame` con una fila por modelo. Es pesado: descarga cada modelo y procesa
todos los audios. **Requiere** que `assets/references/` ya exista (ver
`prepare_references.py`).

In [ ]:
df = asr.run_benchmark()
df

In [ ]:
# 4) Librerías de análisis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## Preparación

Descartamos los modelos que no cargaron (omitidos por presupuesto de RAM, librería
ausente o error) y nos quedamos con las filas evaluadas.

In [ ]:
ok = df[(df["n_audios"] > 0) & (df["load_error"] == "")].copy()
ok = ok.sort_values("wer").reset_index(drop=True)
print(f"{len(ok)} modelos evaluados de {len(df)}")
skipped = df[df["load_error"] != ""]
if len(skipped):
    print("\nOmitidos:")
    print(skipped[["model", "load_error"]].to_string(index=False))
ok[
    [
        "model",
        "family",
        "wer",
        "term_wer",
        "cer",
        "ner_recall",
        "rtf",
        "ram_peak_mb",
        "mean_latency_ms",
        "hallucination_rate",
    ]
]

## Selección del mejor modelo

Score compuesto normalizado `[0, 1]` (mayor = mejor):

- **WER** (menor mejor) — calidad global
- **term-WER** (menor mejor) — calidad en terminología de enfermería
- **RTF** (menor mejor) — velocidad en CPU
- **RAM peak** (menor mejor) — huella de memoria
- **hallucination** (menor mejor) — robustez ante silencio/ruido

Ajusta los pesos según la prioridad del despliegue.

In [ ]:
W = {"wer": 0.35, "term_wer": 0.25, "rtf": 0.20, "ram": 0.10, "halluc": 0.10}


def norm_low(s):  # menor = mejor -> [0,1] con 1 = mejor
    rng = s.max() - s.min()
    return (s.max() - s) / rng if rng else pd.Series(1.0, index=s.index)


ok["score"] = (
    W["wer"] * norm_low(ok["wer"])
    + W["term_wer"] * norm_low(ok["term_wer"])
    + W["rtf"] * norm_low(ok["rtf"])
    + W["ram"] * norm_low(ok["ram_peak_mb"])
    + W["halluc"] * norm_low(ok["hallucination_rate"])
)
ranking = ok.sort_values("score", ascending=False).reset_index(drop=True)
best = ranking.iloc[0]

print(f"🏆 Mejor modelo: {best['model']}  (score = {best['score']:.3f})")
print(
    f"   WER {best['wer']:.3f} · term-WER {best['term_wer']:.3f} · CER {best['cer']:.3f}"
)
print(
    f"   NER recall {best['ner_recall']:.3f} · RTF {best['rtf']:.3f} · "
    f"RAM peak {best['ram_peak_mb']:.0f} MB · halluc {best['hallucination_rate']:.0%}"
)
ranking[
    [
        "model",
        "score",
        "wer",
        "term_wer",
        "cer",
        "ner_recall",
        "rtf",
        "ram_peak_mb",
        "hallucination_rate",
    ]
]

## Imágenes comparativas

Se guardan en `./figures/`.

In [ ]:
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print("guardado:", path)


BASELINE = df[df["is_baseline"]]["model"].iloc[0] if df["is_baseline"].any() else None


def colors_for(rows):
    return ["#e76f51" if m == BASELINE else "#2a9d8f" for m in rows["model"]]

In [ ]:
# WER global vs WER de terminología de enfermería (barras agrupadas)
r = ranking.sort_values("wer", ascending=False)
y = np.arange(len(r))
h = 0.4
fig, ax = plt.subplots()
ax.barh(y + h / 2, r["wer"], h, label="WER global", color="#264653")
ax.barh(
    y - h / 2, r["term_wer"], h, label="WER terminología enfermería", color="#e9c46a"
)
ax.set_yticks(y)
ax.set_yticklabels(r["model"])
ax.set_xlabel("WER (menor es mejor)")
ax.set_title("Calidad: WER global vs terminología clínica")
ax.legend()
save(fig, "01_wer.png")
plt.show()

In [ ]:
# CER por modelo
r = ranking.sort_values("cer", ascending=False)
fig, ax = plt.subplots()
ax.barh(r["model"], r["cer"], color=colors_for(r))
ax.set_xlabel("CER — Character Error Rate (menor es mejor)")
ax.set_title("Error a nivel de carácter")
for i, v in enumerate(r["cer"]):
    ax.text(v, i, f" {v:.3f}", va="center")
save(fig, "02_cer.png")
plt.show()

In [ ]:
# RTF (Real-Time Factor): tiempo de proceso / duración del audio
r = ranking.sort_values("rtf", ascending=False)
fig, ax = plt.subplots()
ax.barh(r["model"], r["rtf"], color=colors_for(r))
ax.axvline(1.0, color="red", ls="--", label="tiempo real (RTF=1)")
ax.set_xlabel("RTF — menor es mejor (<1 = más rápido que el audio)")
ax.set_title("Velocidad de inferencia en CPU")
for i, v in enumerate(r["rtf"]):
    ax.text(v, i, f" {v:.3f}", va="center")
ax.legend()
save(fig, "03_rtf.png")
plt.show()

In [ ]:
# RAM peak vs presupuesto de 2 GB
r = ranking.sort_values("ram_peak_mb")
fig, ax = plt.subplots()
ax.barh(r["model"], r["ram_peak_mb"], color=colors_for(r))
ax.axvline(
    asr.MAX_RAM_GB * 1024,
    color="red",
    ls="--",
    label=f"presupuesto {asr.MAX_RAM_GB:.0f} GB",
)
ax.set_xlabel("RAM peak (MB) — menor es mejor")
ax.set_title("Pico de memoria durante la transcripción")
for i, v in enumerate(r["ram_peak_mb"]):
    ax.text(v, i, f" {v:.0f}", va="center")
ax.legend()
save(fig, "04_ram_peak.png")
plt.show()

In [ ]:
# NER recall: entidades clínicas (meds, valores, abreviaturas) recuperadas
r = ranking.sort_values("ner_recall")
fig, ax = plt.subplots()
ax.barh(r["model"], r["ner_recall"], color="#2a9d8f")
ax.set_xlim(0, 1)
ax.set_xlabel("NER recall (mayor es mejor)")
ax.set_title("Recuperación de terminología clínica")
for i, v in enumerate(r["ner_recall"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "05_ner_recall.png")
plt.show()

In [ ]:
# Rendimiento: latencia media por audio vs duración media del audio (ms)
r = ranking.sort_values("mean_latency_ms", ascending=False)
y = np.arange(len(r))
h = 0.4
fig, ax = plt.subplots()
ax.barh(
    y + h / 2,
    r["mean_latency_ms"],
    h,
    label="latencia media (proceso)",
    color="#e76f51",
)
ax.barh(
    y - h / 2, r["mean_audio_ms"], h, label="duración media del audio", color="#8ecae6"
)
ax.set_yticks(y)
ax.set_yticklabels(r["model"])
ax.set_xlabel("milisegundos")
ax.set_title("Rendimiento: latencia de proceso vs duración del audio")
ax.legend()
save(fig, "06_latency.png")
plt.show()

In [ ]:
# Tasa de alucinación (riesgo de Whisper en silencio/ruido, mitigado con VAD)
r = ranking.sort_values("hallucination_rate", ascending=False)
fig, ax = plt.subplots()
ax.barh(r["model"], r["hallucination_rate"], color="#c1121f")
ax.set_xlabel("Tasa de alucinación (fracción de audios marcados)")
ax.set_title("Alucinación detectada — menor es mejor (VAD + umbral de confianza)")
for i, v in enumerate(r["hallucination_rate"]):
    ax.text(v, i, f" {v:.0%}", va="center")
save(fig, "07_hallucination.png")
plt.show()

In [ ]:
# Trade-off calidad vs velocidad (burbuja = RAM peak, color = score)
fig, ax = plt.subplots()
sizes = (ranking["ram_peak_mb"] / ranking["ram_peak_mb"].max()) * 900 + 80
sc = ax.scatter(
    ranking["rtf"],
    ranking["wer"],
    s=sizes,
    c=ranking["score"],
    cmap="viridis",
    alpha=0.85,
    edgecolors="k",
)
for _, row in ranking.iterrows():
    ax.annotate(
        row["model"],
        (row["rtf"], row["wer"]),
        fontsize=8,
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_xlabel("RTF — menor mejor")
ax.set_ylabel("WER — menor mejor")
ax.set_title("Calidad vs velocidad (burbuja = RAM peak)")
plt.colorbar(sc, label="score compuesto")
save(fig, "08_tradeoff.png")
plt.show()

In [ ]:
# Heatmap de métricas normalizadas (1 = mejor) por modelo
metrics = [
    "wer",
    "term_wer",
    "cer",
    "rtf",
    "ram_peak_mb",
    "hallucination_rate",
    "ner_recall",
]
norm = pd.DataFrame(index=ranking["model"])
for m in metrics:
    s = ranking[m].values
    rng = s.max() - s.min()
    if m == "ner_recall":  # mayor mejor
        norm[m] = (s - s.min()) / rng if rng else 1.0
    else:  # menor mejor
        norm[m] = (s.max() - s) / rng if rng else 1.0
fig, ax = plt.subplots(figsize=(max(8, len(metrics)), 0.6 * len(norm) + 2))
im = ax.imshow(norm.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(metrics, rotation=45, ha="right")
ax.set_yticks(range(len(norm.index)))
ax.set_yticklabels(norm.index)
for i in range(norm.shape[0]):
    for j in range(norm.shape[1]):
        ax.text(j, i, f"{norm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="normalizado (1 = mejor)")
ax.set_title("Métricas normalizadas por modelo")
save(fig, "09_heatmap.png")
plt.show()

In [ ]:
# Ranking final por score compuesto — el mejor resaltado
fig, ax = plt.subplots()
cols = ["#2a9d8f" if i == 0 else "#90be6d" for i in range(len(ranking))]
ax.barh(ranking["model"][::-1], ranking["score"][::-1], color=cols[::-1])
ax.set_xlabel("Score compuesto (mayor = mejor)")
ax.set_title(f"Ranking final  —  🏆 {best['model']}")
for i, v in enumerate(ranking["score"][::-1]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "10_ranking.png")
plt.show()

In [ ]:
# Resumen y recomendación
print("=" * 60)
print("RESUMEN DEL BENCHMARK ASR")
print("=" * 60)
print(f"Audios evaluados   : {int(best['n_audios'])} (16 kHz mono 16-bit)")
print(
    "Ground-truth       : Whisper large-v3 (sesgo declarado, ver prepare_references.py)"
)
print(f"Modelos evaluados  : {len(ok)} de {len(df)}")
print(f"Presupuesto RAM    : {asr.MAX_RAM_GB} GB · VAD anti-alucinación: on")
print("-" * 60)
print(f"RECOMENDADO        : {best['model']}")
print(f"  score compuesto  : {best['score']:.3f}")
print(f"  WER / term-WER   : {best['wer']:.3f} / {best['term_wer']:.3f}")
print(f"  CER              : {best['cer']:.3f}")
print(f"  NER recall       : {best['ner_recall']:.3f}")
print(f"  RTF              : {best['rtf']:.3f}")
print(f"  RAM peak         : {best['ram_peak_mb']:.0f} MB")
print(f"  alucinación      : {best['hallucination_rate']:.0%}")
if BASELINE:
    b = df[df["model"] == BASELINE].iloc[0]
    print("-" * 60)
    print(f"Comparación vs baseline Whisper ({BASELINE}):")
    print(f"  WER  {b['wer']:.3f} -> {best['wer']:.3f}")
    print(f"  RTF  {b['rtf']:.3f} -> {best['rtf']:.3f}")
print("-" * 60)
print(f"Figuras en : {FIG_DIR}")
print(f"Transcripciones en : {asr.OUT_DIR}")